# Vanilla dies vs. Selgis survives

**A/B demo:** one poisoned batch (`loss = loss * float("nan")`) is injected at the
same step into two identical runs on an identical synthetic task.

* **Run A — vanilla PyTorch loop.** The NaN loss flows through the graph, `backward()`
  produces NaN gradients, `optimizer.step()` poisons the weights. Nothing catches it.
* **Run B — `selgis.Trainer`.** The same NaN loss is detected *before* backward,
  the last stable state is restored, the LR is halved and the offending batch is
  skipped. Training continues and finishes.

> Note on the sabotage: we multiply the **real** loss by `float("nan")` so the NaN
> enters the computation graph. A bare `torch.tensor(float("nan"))` that never
> touches the graph would *not* poison the weights.

CPU-only, runs in a few seconds. Every claim below is backed by an explicit `assert`.

In [1]:
import logging
import sys
import tempfile
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*IProgress not found.*")


def find_repo_root() -> Path:
    """Locate the selgis checkout so the demo runs the local code,
    not a pip-installed copy, regardless of the kernel's cwd."""
    try:
        candidates = [Path(__file__).resolve().parent]
    except NameError:
        candidates = []
    candidates.extend([Path.cwd(), *Path.cwd().resolve().parents])
    for cand in candidates:
        if (cand / "selgis" / "__init__.py").exists():
            return cand
    raise RuntimeError("selgis package not found near this notebook")


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from selgis import SelgisConfig, Trainer
from selgis.utils import seed_everything

import selgis
print("repo root:", ROOT)
print("selgis:", selgis.__version__, "-", selgis.__file__)

repo root: C:\Selgis ML2.0
selgis: 0.2.7.3 - C:\Selgis ML2.0\selgis\__init__.py


## The task and the sabotage

A tiny 2-class classifier on 128 synthetic samples (8 batches per epoch, fixed
order, no shuffling). Both runs start from **identical weights** and see
**identical batches**. `SabotageLoss` multiplies the real loss by NaN exactly
once — on the same call in both runs — and records every loss it produced.

In [2]:
seed_everything(42)

X = torch.randn(128, 10)
y = (X[:, 0] + X[:, 1] > 0).long()  # trivially separable
loader = DataLoader(TensorDataset(X, y), batch_size=16, shuffle=False)

EPOCHS = 5
STEPS_PER_EPOCH = len(loader)          # 8
POISON_STEP = 20                        # 0-based global step in both runs


class SabotageLoss(nn.Module):
    """CrossEntropyLoss that poisons the REAL loss of exactly one batch."""

    def __init__(self, poison_call: int):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.poison_call = poison_call
        self.calls = 0
        self.values: list[float] = []
        self.poisoned = False

    def forward(self, logits, labels):
        loss = self.ce(logits, labels)
        self.calls += 1
        if self.calls == self.poison_call:
            loss = loss * float("nan")  # NaN enters the computation graph here
            self.poisoned = True
        self.values.append(float(loss))
        return loss


def build_model() -> nn.Module:
    torch.manual_seed(7)
    return nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))


ref_model = build_model()
INIT_STATE = {k: v.clone() for k, v in ref_model.state_dict().items()}
print(f"task: 128 samples, batch 16, {STEPS_PER_EPOCH} steps/epoch; "
      f"sabotage on step {POISON_STEP} (of {EPOCHS * STEPS_PER_EPOCH})")

task: 128 samples, batch 16, 8 steps/epoch; sabotage on step 20 (of 40)


## Run A — vanilla PyTorch loop

A plain training loop, the kind everyone writes by hand. Watch the loss at the
sabotaged step and after it.

In [3]:
sab_a = SabotageLoss(poison_call=POISON_STEP + 1)  # criterion call == step + 1
model_a = build_model()
model_a.load_state_dict(INIT_STATE)               # identical starting weights

optimizer = torch.optim.AdamW(model_a.parameters(), lr=1e-2)
step = 0
for _epoch in range(EPOCHS):
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = sab_a(model_a(xb), yb)
        loss.backward()          # sabotaged step -> NaN gradients
        optimizer.step()         # -> NaN weights, with no one to notice
        step += 1

vanilla_losses = sab_a.values
vanilla_final_loss = vanilla_losses[-1]
vanilla_weights_nan = any(torch.isnan(p).any() for p in model_a.parameters())

print(f"steps around the sabotage:")
for i in range(POISON_STEP - 1, min(POISON_STEP + 4, len(vanilla_losses))):
    print(f"  step {i:>2}: loss={vanilla_losses[i]:.4f}")
print(f"final train loss: {vanilla_final_loss}")
print(f"NaN in weights:   {vanilla_weights_nan}")

assert sab_a.poisoned, "sabotage must have fired"
assert any(v != v for v in vanilla_losses[POISON_STEP:]), (
    "vanilla losses after the sabotage must be NaN"
)
assert vanilla_weights_nan, "vanilla weights must be poisoned by the NaN batch"
print()
print("VERDICT: VANILLA DIED: loss is NaN, weights are NaN, "
      f"{EPOCHS * STEPS_PER_EPOCH - POISON_STEP} steps of compute produced garbage")

steps around the sabotage:
  step 19: loss=0.2001
  step 20: loss=nan
  step 21: loss=nan
  step 22: loss=nan
  step 23: loss=nan
final train loss: nan
NaN in weights:   True

VERDICT: VANILLA DIED: loss is NaN, weights are NaN, 20 steps of compute produced garbage


## Run B — Selgis

Same task, same weights, same sabotage on the same step. `SelgisCore.check_loss()`
inspects the loss **before** backward: NaN → rollback to the last stable state,
optimizer moments zeroed, LR halved, the offending batch is skipped. A logging
handler captures the rollback so we can prove it happened.

In [4]:
class LogCollector(logging.Handler):
    def __init__(self):
        super().__init__()
        self.messages: list[str] = []

    def emit(self, record):
        self.messages.append(record.getMessage())


sab_b = SabotageLoss(poison_call=POISON_STEP + 1)
model_b = build_model()
model_b.load_state_dict(INIT_STATE)               # identical starting weights

with tempfile.TemporaryDirectory() as tmp_dir:
    config = SelgisConfig(
        max_epochs=EPOCHS,
        batch_size=16,
        learning_rate=1e-2,
        nan_recovery=True,
        state_update_interval=5,   # snapshot every 5 optimizer steps
        logging_steps=1000,        # keep the log clean; we collect WARNINGS anyway
        state_storage="memory",
        output_dir=tmp_dir,
    )
    collector = LogCollector()
    selgis_logger = logging.getLogger("selgis")
    selgis_logger.addHandler(collector)
    selgis_logger.setLevel(logging.WARNING)
    try:
        trainer = Trainer(
            model_b,
            config,
            train_dataloader=loader,
            eval_dataloader=None,
            criterion=sab_b,
        )
        metrics = trainer.train()
    finally:
        selgis_logger.removeHandler(collector)

selgis_losses = sab_b.values
selgis_weights_nan = any(torch.isnan(p).any() for p in model_b.parameters())
rollback_logged = any("Rollback triggered" in m for m in collector.messages)

print(f"loss around the sabotage:")
for i in range(POISON_STEP - 1, min(POISON_STEP + 4, len(selgis_losses))):
    print(f"  step {i:>2}: loss={selgis_losses[i]:.4f}")
print(f"final train loss: {metrics['train_loss']:.4f}")
print(f"NaN in weights:   {selgis_weights_nan}")
print("rollback log entries:")
for m in collector.messages:
    print(f"  [selgis] {m}")

assert sab_b.poisoned, "the same sabotage must have fired in run B"
assert selgis_losses[POISON_STEP] != selgis_losses[POISON_STEP], (
    "the sabotaged step must have produced a NaN loss"
)
assert not selgis_weights_nan, "selgis weights must stay finite"
assert metrics["train_loss"] == metrics["train_loss"], "train_loss must not be NaN"
assert all(v == v for v in selgis_losses[POISON_STEP + 1:]), (
    "training must continue producing finite losses after the rollback"
)
assert rollback_logged, "a rollback must appear in the log"
print()
print(f"VERDICT: SELGIS SURVIVED: final loss {metrics['train_loss']:.4f}, "
      "weights finite, recovery was automatic")

loss around the sabotage:
  step 19: loss=0.2011
  step 20: loss=nan
  step 21: loss=0.3087
  step 22: loss=0.2479
  step 23: loss=0.3948
final train loss: 0.1720
NaN in weights:   False
rollback log entries:
  [selgis] Rollback triggered: NaN/Inf loss detected

VERDICT: SELGIS SURVIVED: final loss 0.1720, weights finite, recovery was automatic


## Side by side

| | Vanilla loop | Selgis |
|---|---|---|
| sabotaged step | NaN loss | NaN loss detected **before** backward |
| what happened | NaN gradients → NaN weights | rollback + LR halved + batch skipped |
| final train loss | `nan` | finite (see above) |
| weights after run | poisoned | finite |
| human intervention at 3 AM | required | none |

In [5]:
finite = lambda v: v == v

print(f"{'':24}{'vanilla':>14}{'selgis':>14}")
print(f"{'final train loss':24}{vanilla_final_loss:>14.4f}{metrics['train_loss']:>14.4f}")
print(f"{'weights all finite':24}{str(not vanilla_weights_nan):>14}{str(not selgis_weights_nan):>14}")
print(f"{'losses finite after NaN':24}{str(all(finite(v) for v in vanilla_losses[POISON_STEP + 1:])):>14}"
      f"{str(all(finite(v) for v in selgis_losses[POISON_STEP + 1:])):>14}")
print(f"{'automatic rollbacks':24}{0:>14}{sum('Rollback triggered' in m for m in collector.messages):>14}")

print()
print("=" * 60)
print("VERDICT: vanilla run is DEAD (NaN loss, NaN weights).")
print("VERDICT: selgis run SURVIVED (finite loss, finite weights,")
print(f"         {sum('Rollback triggered' in m for m in collector.messages)} automatic rollback).")
print("=" * 60)

                               vanilla        selgis
final train loss                   nan        0.1720
weights all finite               False          True
losses finite after NaN          False          True
automatic rollbacks                  0             1

VERDICT: vanilla run is DEAD (NaN loss, NaN weights).
VERDICT: selgis run SURVIVED (finite loss, finite weights,
         1 automatic rollback).
